In [150]:
import warnings 
warnings.filterwarnings('ignore')


In [151]:
# -------------------
# Document load
# -------------------

import langchain_community 
from langchain_community.document_loaders import PyPDFLoader,WebBaseLoader
urls = [
    "https://en.wikipedia.org/wiki/India",
    "https://en.wikipedia.org/wiki/History_of_India",
    "https://en.wikipedia.org/wiki/Geography_of_India",
    "https://en.wikipedia.org/wiki/Politics_of_India",
    "https://en.wikipedia.org/wiki/Economy_of_India",
    "https://en.wikipedia.org/wiki/Demographics_of_India",
    "https://en.wikipedia.org/wiki/Culture_of_India",
    "https://en.wikipedia.org/wiki/Languages_of_India",
    "https://en.wikipedia.org/wiki/States_and_union_territories_of_India",
    "https://en.wikipedia.org/wiki/Constitution_of_India",
    "https://en.wikipedia.org/wiki/Military_of_India",
    "https://en.wikipedia.org/wiki/Science_and_technology_in_India",
    "https://en.wikipedia.org/wiki/Sport_in_India",
    "https://en.wikipedia.org/wiki/Education_in_India",
    "https://en.wikipedia.org/wiki/Health_in_India",
    "https://en.wikipedia.org/wiki/Virat_Kohli"
]

loder = WebBaseLoader(urls)
pages = loder.load()


In [152]:
# ------------------
# Split the text
# ------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter
spliter = RecursiveCharacterTextSplitter(chunk_size = 4000,chunk_overlap = 1000)

texts = spliter.split_documents(pages)

# ---------------
# chunking
# ---------------

chunks = []
for i in texts:
    chunks.append(i.page_content)

# ---------------
# Metadata stord
# ---------------

metadata = []

for i in texts:
    metadata.append(i.metadata)


print(f'first 5 chunks : {chunks[:5]}')


first 5 chunks : ['India - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nEtymology\n\n\n\n\n\n\n\n\n2\nHistory\n\n\n\n\nToggle History subsection\n\n\n\n\n\n2.1\nAncient India\n\n\n\n\n\n\n\n\n2.2\nMedieval India\n\n\n\n\n\n\n\n\n2.3\nEarly modern India\n\n\n\n\n\n\n\n\n2.4\nMod

In [153]:
# --------------------------------
# Embedding and vector stord
# --------------------------------

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding_function = SentenceTransformerEmbeddingFunction('all-MiniLM-L6-v2')

# -------------
# client
# -------------

client = chromadb.PersistentClient(path='./Multi_Agent-DB')
collection = client.get_or_create_collection(name='CodeBase',embedding_function=embedding_function)


In [154]:
import hashlib

# ids = []
# for i in chunks:
#     encoded = i.encode('utf-8')
#     hash_obj = hashlib.md5(encoded)
#     hex_id = hash_obj.hexdigest()
#     ids.append(hex_id)

# -----------------------
# create an unique id's
# -----------------------
ids = [hashlib.md5(i.encode('utf-8')).hexdigest() for i in chunks]
seen = set()
unique_ids = []
unique_chunks = []
unique_metadata = []

for id_, chunk, meta in zip(ids, chunks, metadata):
    if id_ not in seen:
        seen.add(id_)
        unique_ids.append(id_)
        unique_chunks.append(chunk)
        unique_metadata.append(meta)

ids = unique_ids
chunks = unique_chunks
metadata = unique_metadata
if chunks:
    collection.upsert(
        ids = ids,
        documents=chunks,
        metadatas=metadata
    )
print(f'print the total chunks : {collection.count()}')


print the total chunks : 1281


In [155]:
# ------------------------
# create the token
# ------------------------

from rank_bm25 import BM25Okapi

def token_create(i:str):
    i = i.lower()
    i = i.split()
    return i
token = [token_create(i) for i in chunks]
token_for_keyword = BM25Okapi(token)


In [156]:
# ----------------------
# create calculetor tool
# ----------------------

from langchain_core.tools import tool

import ast

@tool
def calculetor(exec:str):
    """ this tools for anytype of arithmetic operation """
    try:
        result = ast.literal_eval(exec)
        return result
    except Exception as e:
        print(str(e))


In [157]:
# -------------------
# create weather tool
# -------------------

import requests

@tool
def weather(location:str):
    """ Just provide the curennt weather of user provided location """
    try:
        weather = requests.get(f"https://wttr.in/{location}?format=3")
        return weather.text
    except Exception as e:
        print(str(e))


In [158]:
# -------------------------
# create the chat model

import os 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

api = os.getenv('GRQO_API_KEY')

chat_model = ChatGroq(model='openai/gpt-oss-20b',api_key=api,temperature=0)


In [159]:
# -----------------
# Retrival tool
# -----------------

@tool
def retrival_for_local_docs(query:str):
    """ every user asking questions reach through this tool """
    promot = f""" Rewrite the query based on symentic search : {query} """

    query_rewrite = chat_model.invoke(promot).content

    result = collection.query(query_texts=[query_rewrite],n_results=5)
    documents = result['documents'][0]
    distance = result['distances'][0]

    thresold = 0.89

    #----------------
    # near_chunks find
    # ---------------

    near_chunks = []

    for i , doc in zip(distance,documents):
        if thresold>i:
            near_chunks.append(doc)

    score = token_for_keyword.get_scores(token_create(query_rewrite))

    # -----------------------------
    # find the near index/chunk
    # -----------------------------

    def find_near_index(score,k=10):
        index = list(enumerate(score))
        sorted_index = sorted(index,key=lambda x:x[1],reverse=True)
        return [inx for inx , sc in sorted_index[:k]]
    get_near_index = find_near_index(score,k=10)

    # ------------------------------------------
    # find the doc value help of the index
    # ------------------------------------------

    index_to_chunk = []
    for doc in get_near_index:
        index_to_chunk.append(chunks[doc])

    # -----------------------
    # create rrf_items
    # ------------------------

    rrf_item = {}

    for rank , doc in enumerate(near_chunks):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_to_chunk):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)

    # ----------------------
    # merge the rrf_item
    # -----------------------

    merge = sorted(rrf_item.items(),key=lambda x:x[1],reverse=True)

    # ----------------
    # find top doc
    # ----------------
    top_chunks = []

    for doc , _ in merge[:5]:
        top_chunks.append(doc)


    if top_chunks:
        return "\n\n".join(top_chunks)
    return "NOT RELATED CONTENT"


    


In [160]:
# -----------------------------
# bind with anget and tool
# -----------------------------

calculetor_agent = chat_model.bind_tools([calculetor])
weather_agent = chat_model.bind_tools([weather])
retrival_agent = chat_model.bind_tools([retrival_for_local_docs])

tools = [calculetor,weather,retrival_for_local_docs]

tool_name = {t.name : t for t in tools}

print(tool_name)



{'calculetor': StructuredTool(name='calculetor', description='this tools for anytype of arithmetic operation', args_schema=<class 'langchain_core.utils.pydantic.calculetor'>, func=<function calculetor at 0x000002DD23FDC2C0>), 'weather': StructuredTool(name='weather', description='Just provide the curennt weather of user provided location', args_schema=<class 'langchain_core.utils.pydantic.weather'>, func=<function weather at 0x000002DD23FDC360>), 'retrival_for_local_docs': StructuredTool(name='retrival_for_local_docs', description='every user asking questions reach through this tool', args_schema=<class 'langchain_core.utils.pydantic.retrival_for_local_docs'>, func=<function retrival_for_local_docs at 0x000002DD0AF7B4C0>)}


In [161]:
agents = {
    "calculetor_agent" : calculetor_agent,
    "weather_agent" : weather_agent,
    "retrival_agent" : retrival_agent
}

from langchain_core.messages import SystemMessage,HumanMessage

agentic_prompt = {
    "calculetor_agent" : SystemMessage(content="you are a calculator tool so useful to do arithmetic operations"),
    "weather_agent": SystemMessage(content=" you are a waether tool Just provide the curennt weather of user provided location"),
    "retrival_agent": SystemMessage(content="are a reliable research assistance so asking qustion always ask on the local document then transfer it to websearch")

}

print(tool_name)


{'calculetor': StructuredTool(name='calculetor', description='this tools for anytype of arithmetic operation', args_schema=<class 'langchain_core.utils.pydantic.calculetor'>, func=<function calculetor at 0x000002DD23FDC2C0>), 'weather': StructuredTool(name='weather', description='Just provide the curennt weather of user provided location', args_schema=<class 'langchain_core.utils.pydantic.weather'>, func=<function weather at 0x000002DD23FDC360>), 'retrival_for_local_docs': StructuredTool(name='retrival_for_local_docs', description='every user asking questions reach through this tool', args_schema=<class 'langchain_core.utils.pydantic.retrival_for_local_docs'>, func=<function retrival_for_local_docs at 0x000002DD0AF7B4C0>)}


In [162]:
# -------------------------------
# find the agent / router tool
# --------------------------------
from langchain_core.messages import AIMessage
def agent_find(query:str):
    prompt = f""" select the tool based on user given qustion . One tool at a time {query} 
    answers should be like : calculator_agent , weather_agent , retrival_agent """

    response = chat_model.invoke(prompt).content.strip().lower()

    if "calculetor_agent" in response:
        return "calculetor_agent"
    elif "weather_agent" in response:
        return "weather_agent"
    elif "retrival_agent" in response:
        return "retrival_agent"
    else:
        return "NOT RELATED CONTENT"



# --------------------------------
# search and answer genarate
# --------------------------------

def search(query:str , state: dict = None):
    if state is None:
        state = {'history' : []}

    label = agent_find(query=query) 

    if label == "NOT RELATED CONTENT":
        answer = chat_model.invoke(query).content
        state['history'].append({'query':query , 'answer':answer})
        return answer ,state

    agent = agents[label]
    system_prompt = agentic_prompt[label]
    messages = [system_prompt]
    for turn in state['history'][-5:]:
        messages.append(HumanMessage(content=turn['query']))
        messages.append(AIMessage(content=str(turn['answer'])))
    messages.append(HumanMessage(content=query))

    response = agent.invoke(messages)

    if not response.tool_calls:
        state['history'].append({'query':query,'answer':response.content})
        return response.content , state

    for call in response.tool_calls:
        args = call['args']
        tool_names = call['name']
        tool_output = tool_name[tool_names].invoke(args)
        state['history'].append({'query':query,'answer':tool_output})
        return tool_output , state


In [163]:
question = 'what is 56*90??'
content = search(query=question)
print(content)


('56\u202f×\u202f90\u202f=\u202f5,040.', {'history': [{'query': 'what is 56*90??', 'answer': '56\u202f×\u202f90\u202f=\u202f5,040.'}]})
